# Figure 3 — SLDA vs DLBT Schematic

Each cell draws one visual element. Run all cells, then the final assembly cell to export the SVG.

**Macro-knobs** live in the first cell (`CFG`). Adjust them to fine-tune spacing, colours, font sizes, etc. without touching the drawing logic.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 0 — Global config / macro-knobs
# ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, Polygon as MplPoly, FancyArrowPatch
from scipy.special import gammaln

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica Neue', 'Arial', 'DejaVu Sans']

CFG = dict(
    # ── canvas ──────────────────────────────────────────────────
    fig_w       = 22,       # figure width  (inches)
    fig_h       = 14,       # figure height (inches)

    # ── column geometry (in axes-fraction of figure) ─────────────
    col_gap     = 0.04,     # horizontal gap between the two panel boxes
    box_pad     = 0.015,    # inner padding inside each panel box
    top_strip   = 0.10,     # height reserved for shared stimulus row
    bot_strip   = 0.05,     # height reserved for P̂(yes|t,x) labels

    # ── colours ──────────────────────────────────────────────────
    slda_color      = '#6a3d9a',   # purple  — SLDA border
    dlbt_color      = '#cb2027',   # red     — DLBT border
    box_lw          = 2.5,         # border linewidth
    box_rounding    = 0.02,        # FancyBboxPatch rounding (figure-fraction)

    clip_box_fc  = 'black',
    clip_box_ec  = 'black',
    clip_box_tc  = 'white',

    task_colors  = ['#1b9e77', '#d95f02', '#7570b3'],  # green, orange, purple
    task_edges   = ['#147b5c', '#b34d01', '#5a548f'],
    task_names   = ['Task 1', 'Task 2', 'Task T'],
    task_k       = [0.8, 2.5, 1.4],                   # sigmoid steepness per task

    # ── font sizes ───────────────────────────────────────────────
    fs_title     = 16,     # panel box title (SLDA / DLBT)
    fs_label     = 13,     # axes labels inside clouds
    fs_task      = 11,     # per-task box title
    fs_desc      = 11,     # descriptive italic text
    fs_phat      = 11,     # P̂(yes|t,x) labels
    fs_box       = 13,     # Attpool / Mapper / Clip box text

    # ── point-cloud appearance ───────────────────────────────────
    cloud_seed   = 13,
    cloud_n      = 600,
    cloud_s      = 14,
    cloud_alpha  = 0.18,
    cloud_color  = '0.35',

    # ── dirichlet surface ─────────────────────────────────────────
    dir_alpha    = np.array([4.0, 4.0, 3.0]),
    dir_n_strips = 160,
    dir_n_sub    = 100,
    dir_d_scale  = 0.95,
    dir_cmap     = 'YlOrRd',
    dir_plane_c  = '#4CAF50',
    dir_plane_a  = 0.24,
    dir_z2c      = 0.38,
)

# Derived colour shortcuts
SLDA_C = CFG['slda_color']
DLBT_C = CFG['dlbt_color']
print('Config loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 1 — Shared helper functions
# ─────────────────────────────────────────────────────────────────

def arrow(ax, start, end, color='black', lw=1.3, ms=10, zorder=10,
          style='->', shrinkA=0, shrinkB=0):
    """Annotate-based arrow from start to end."""
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle=style, lw=lw, color=color,
                                shrinkA=shrinkA, shrinkB=shrinkB,
                                mutation_scale=ms),
                zorder=zorder)


def black_label_box(ax, xy, text, fs=None, pad=0.18, fc='black', tc='white',
                    rounding=0.1, zorder=5, width=None):
    """Draw a filled rounded-rect label box centred at xy (axes coords)."""
    fs = fs or CFG['fs_box']
    t = ax.text(xy[0], xy[1], text,
                fontsize=fs, color=tc, fontweight='bold',
                ha='center', va='center', zorder=zorder + 1,
                transform=ax.transAxes)
    t.set_bbox(dict(facecolor=fc, edgecolor=fc,
                    boxstyle=f'round,pad={pad}',
                    zorder=zorder))
    return t


def draw_cloud(ax, rng=None, n=None, s=None, alpha=None, color=None, xlim=None, ylim=None):
    """Draw the CLIP feature-space point cloud."""
    rng   = rng   or np.random.default_rng(CFG['cloud_seed'])
    n     = n     or CFG['cloud_n']
    s     = s     or CFG['cloud_s']
    alpha = alpha or CFG['cloud_alpha']
    color = color or CFG['cloud_color']
    mean = np.array([0.72, 0.95])
    cov  = np.array([[0.55, 0.16], [0.16, 0.40]])
    X = rng.multivariate_normal(mean, cov, size=n)
    ax.scatter(X[:, 0], X[:, 1], s=s, color=color, alpha=alpha,
               linewidths=0, zorder=3)
    return X


# Standard axis arrows for the CLIP feature-space sub-axes
_ORIGIN  = np.array([0.0,  0.0])
_AXES_CLOUD = [
    (np.array([ 0.00,  3.15]), r'$d_1$',     ( 0.00,  0.28)),
    (np.array([ 3.35,  0.00]), r'$d_2$',     ( 0.28,  0.02)),
    (np.array([ 2.45, -0.55]), r'$d_3$',     ( 0.28, -0.03)),
    (np.array([ 1.75, -0.95]), r'$d_4$',     ( 0.22, -0.14)),
    (np.array([-0.90, -1.30]), r'$d_{1024}$',(-0.38, -0.18)),
]

def draw_cloud_axes(ax, fs=None):
    """Draw the 5 pseudo-3-D axis arrows for the CLIP latent-space panel."""
    fs = fs or CFG['fs_label']
    for end, label, off in _AXES_CLOUD:
        arrow(ax, _ORIGIN, end, lw=1.2, ms=9, zorder=10)
        ax.text(end[0] + off[0], end[1] + off[1], label,
                fontsize=fs, ha='center', va='center', zorder=11)
    ax.scatter([0.18, 0.38, 0.58], [-1.42]*3, s=6, color='black', zorder=11)


def cloud_ax_style(ax):
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-2.55, 4.05)
    ax.set_ylim(-1.85, 3.65)


print('Helpers loaded.')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 2 — Element: CLIP latent-space point cloud
# (standalone preview — axes-level drawing function used in assembly)
# ─────────────────────────────────────────────────────────────────

def draw_latent_space(ax, rng=None, title=None, title_fs=None, annotation=None):
    """
    Draw the CLIP feature-space panel into `ax`.
    ax   : a matplotlib Axes already created by the caller
    rng  : numpy RNG (reproducible cloud)
    title: optional string shown above (e.g. 'CLIP feature space f(x)∈ℝ¹⁰²⁴')
    """
    rng = rng or np.random.default_rng(CFG['cloud_seed'])
    cloud_ax_style(ax)
    draw_cloud(ax, rng=rng)
    draw_cloud_axes(ax)

    if title:
        fs = title_fs or CFG['fs_desc']
        ax.text(0.5, 1.04, title, transform=ax.transAxes,
                ha='center', va='bottom', fontsize=fs,
                fontweight='bold')

    if annotation:   # e.g. teal circle for 'this stimulus'
        ax.scatter([annotation[0]], [annotation[1]],
                   s=90, facecolors='none', edgecolors='#00b0b9',
                   linewidths=1.8, zorder=12)


# ── preview ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, 3.8))
draw_latent_space(ax,
                  title=r'CLIP feature space $f(x) \in \mathbb{R}^{1024}$',
                  annotation=(0.72, 1.95))
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 3 — Element: per-task hyperplane panel (cloud + slicing plane)
# ─────────────────────────────────────────────────────────────────

def _make_plane(center, u, v):
    c, u, v = np.array(center), np.array(u), np.array(v)
    return np.array([c-u-v, c+u-v, c+u+v, c-u+v])

_PLANES = {
    'Task 1': _make_plane((0.55, 0.65), ( 2.35,  0.32), ( 0.22,  1.10)),
    'Task 2': _make_plane((0.85, 0.95), ( 1.90, -0.82), ( 0.96,  0.28)),
    'Task T': _make_plane((0.40, 0.80), ( 1.30,  1.48), (-1.18,  0.32)),
}


def draw_task_panel(ax_cloud, name, color, edge, rng=None, plane_alpha=0.30,
                    cloud_title=False, fs_task=None, fs_label=None):
    """
    Draw the per-task cloud+hyperplane into ax_cloud.
    A rounded coloured border is added directly on ax_cloud.
    """
    rng      = rng or np.random.default_rng(CFG['cloud_seed'] + hash(name) % 1000)
    fs_task  = fs_task  or CFG['fs_task']
    fs_label = fs_label or max(CFG['fs_label'] - 5, 7)

    cloud_ax_style(ax_cloud)
    draw_cloud(ax_cloud, rng=rng, s=CFG['cloud_s'] - 4, alpha=CFG['cloud_alpha'])

    # hyperplane
    ax_cloud.add_patch(MplPoly(_PLANES[name], closed=True,
                               facecolor=color, edgecolor=edge,
                               alpha=plane_alpha, linewidth=1.1,
                               joinstyle='round', zorder=2))

    draw_cloud_axes(ax_cloud, fs=fs_label)

    # coloured border + title
    ax_cloud.set_facecolor('none')
    for sp in ax_cloud.spines.values():
        sp.set_visible(False)
    ax_cloud.add_patch(FancyBboxPatch(
        (0, 0), 1, 1,
        boxstyle='round,pad=0,rounding_size=0.06',
        transform=ax_cloud.transAxes,
        facecolor=color + '22', edgecolor=edge,
        linewidth=1.6, clip_on=False, zorder=0))
    ax_cloud.set_title(name, color=edge, fontsize=fs_task,
                       fontweight='bold', pad=4)


# ── preview: three task panels side by side ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(7, 3.5))
rng_base = np.random.default_rng(CFG['cloud_seed'])
for ax, name, color, edge in zip(axes, CFG['task_names'],
                                  CFG['task_colors'], CFG['task_edges']):
    draw_task_panel(ax, name, color, edge, rng=rng_base)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 4 — Element: logistic sigmoid curve
# ─────────────────────────────────────────────────────────────────
import seaborn as sns

def draw_sigmoid(ax, color, k=1.0, xlim=(-4.5, 4.5),
                 lw=2.0, vline=True):
    """
    Draw a sigmoid σ(k·z) into ax.
    k   : steepness (task-specific macro-knob)
    """
    z = np.linspace(xlim[0], xlim[1], 400)
    p = 1.0 / (1.0 + np.exp(-k * z))
    ax.plot(z, p, color=color, lw=lw)
    if vline:
        ax.axvline(0, color='0.55', lw=0.8, ls='--')

    ax.set_xlim(*xlim)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([])
    ax.set_yticks([])

    # schematic y-axis arrow
    ax.annotate('', xy=(0, 1.08), xytext=(0, -0.08),
                xycoords=('axes fraction', 'data'),
                textcoords=('axes fraction', 'data'),
                arrowprops=dict(arrowstyle='-|>', color='0.30',
                                lw=0.9, mutation_scale=8),
                annotation_clip=False)
    ax.text(-0.18, 0.0, '0', transform=ax.get_yaxis_transform(),
            fontsize=8, ha='right', va='center', color='0.40')
    ax.text(-0.18, 1.0, '1', transform=ax.get_yaxis_transform(),
            fontsize=8, ha='right', va='center', color='0.40')

    # schematic x-axis arrow
    ax.annotate('', xy=(1.08, 0), xytext=(-0.08, 0),
                xycoords=('axes fraction', 'data'),
                textcoords=('axes fraction', 'data'),
                arrowprops=dict(arrowstyle='-|>', color='0.30',
                                lw=0.9, mutation_scale=8),
                annotation_clip=False)

    sns.despine(ax=ax, top=True, right=True, bottom=True, left=True)


# ── preview ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for ax, color, k in zip(axes, CFG['task_colors'], CFG['task_k']):
    draw_sigmoid(ax, color=color, k=k)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 5 — Element: Dirichlet surface (with optional hyperplane)
# ─────────────────────────────────────────────────────────────────

def _logpdf(bary, alpha):
    log_B = np.sum(gammaln(alpha)) - gammaln(alpha.sum())
    return np.sum((alpha - 1.0) * np.log(np.clip(bary, 1e-300, None)), axis=1) - log_B


def _soften(rgba, mix=(1,1,1), amount=0.0):
    rgb = np.array(rgba[:3])
    return (*((1-amount)*rgb + amount*np.array(mix)), rgba[3])


_DIR_PROJ = dict(
    e_dens = np.array([ 0.00,  3.15]),
    e_z1   = np.array([ 3.35,  0.00]),
    e_z2   = np.array([-1.80, -1.20]),
    e_z3   = np.array([ 1.75, -0.95]),
)
_DIR_AXES = [
    ('e_dens', 'density',  ( 0.00,  0.30), 13, True ),
    ('e_z1',  r'$z_1$',   ( 0.28,  0.02), 18, False),
    ('e_z3',  r'$z_2$',   ( 0.28, -0.12), 18, False),
    (None,    r'$z_3$',   ( 0.25, -0.18), 18, False),   # fixed endpoint below
    (None,    r'$z_K$',   (-0.38, -0.10), 18, False),
]
_DIR_FIXED = {
    r'$z_3$': np.array([ 0.55, -1.25]),
    r'$z_K$': np.array([-1.30, -0.80]),
}


def draw_dirichlet(ax, alpha=None, n_strips=None, n_sub=None,
                   d_scale=None, cmap=None,
                   show_plane=True, plane_color=None, plane_alpha=None,
                   z2c=None, fs=None):
    """
    Render a pseudo-3-D Dirichlet density surface into ax.
    All parameters fall back to CFG values if omitted.
    show_plane : whether to draw the green decision-cut hyperplane
    """
    alpha       = alpha       if alpha       is not None else CFG['dir_alpha']
    n_strips    = n_strips    or CFG['dir_n_strips']
    n_sub       = n_sub       or CFG['dir_n_sub']
    d_scale     = d_scale     or CFG['dir_d_scale']
    cmap        = cmap        or CFG['dir_cmap']
    plane_color = plane_color or CFG['dir_plane_c']
    plane_alpha = plane_alpha if plane_alpha is not None else CFG['dir_plane_a']
    z2c         = z2c         or CFG['dir_z2c']
    fs          = fs          or CFG['fs_label']

    e_dens, e_z1, e_z2, e_z3 = (_DIR_PROJ[k] for k in
                                  ('e_dens','e_z1','e_z2','e_z3'))

    # global pdf max for colour normalisation
    tg = np.linspace(0.005, 0.995, 80)
    g1, g2 = np.meshgrid(tg, tg)
    ok = (g1 + g2) <= 0.995
    b_all = np.column_stack([g1[ok], g2[ok], 1-g1[ok]-g2[ok]])
    lpdf_max = _logpdf(b_all, alpha).max()

    cmap_fn = plt.get_cmap(cmap)
    origin  = np.array([0.0, 0.0])

    ax.set_aspect('equal')
    ax.axis('off')

    # axis arrows
    for key, label, off, fsize, italic in _DIR_AXES:
        end = _DIR_PROJ[key] if key else _DIR_FIXED[label]
        arrow(ax, origin, end, lw=1.45, ms=13, zorder=1)
        ax.text(end[0]+off[0], end[1]+off[1], label,
                fontsize=fsize, ha='center', va='center', zorder=1,
                style='italic' if italic else 'normal')
    ax.scatter([-0.05,-0.20,-0.35], [-1.38]*3, s=10, color='black', zorder=1)

    # build strips
    z2_vals = np.linspace(0.005, 0.975, n_strips)
    strips  = []
    for z2v in sorted(z2_vals, reverse=True):
        z1_max = 1.0 - z2v - 0.005
        if z1_max < 0.02:
            continue
        z1   = np.linspace(0.005, z1_max, n_sub+1)
        z3   = 1.0 - z1 - z2v
        bary = np.column_stack([z1, np.full_like(z1, z2v), z3])
        pdf  = np.exp(_logpdf(bary, alpha) - lpdf_max)
        dv   = pdf * d_scale
        sx_f = z1*e_z1[0] + z2v*e_z2[0] + z3*e_z3[0]
        sy_f = z1*e_z1[1] + z2v*e_z2[1] + z3*e_z3[1]
        sx   = sx_f + dv*e_dens[0]
        sy   = sy_f + dv*e_dens[1]
        strips.append((z2v, z1, z3, pdf, sx_f, sy_f, sx, sy))

    # pass 1: all strips (back → front)
    for z2v, z1, z3, pdf, sx_f, sy_f, sx, sy in strips:
        is_front = z2v < z2c
        for k in range(n_sub):
            avg = 0.5*(pdf[k]+pdf[k+1])
            c   = cmap_fn(avg)
            if is_front:
                c = _soften(c, (1,.45,.45), 0.12);  a_q = 0.12
            else:
                c = _soften(c, (1,1,1), 0.18);       a_q = 0.085
            ax.add_patch(MplPoly(list(zip(
                [sx[k],sx[k+1],sx_f[k+1],sx_f[k]],
                [sy[k],sy[k+1],sy_f[k+1],sy_f[k]])),
                facecolor=c, edgecolor='none', linewidth=0, alpha=a_q, zorder=2))

    # pass 2: decision-cut hyperplane
    if show_plane:
        def _fp(z1v):
            z3v = 1-z1v-z2c
            return np.array([z1v*e_z1[0]+z2c*e_z2[0]+z3v*e_z3[0],
                              z1v*e_z1[1]+z2c*e_z2[1]+z3v*e_z3[1]])
        t_lo, t_hi = -0.45, 1.15
        p_bl = _fp(t_lo); p_br = _fp(t_hi)
        h    = d_scale * e_dens * 1.15
        pts  = [p_bl, p_br, p_br+h, p_bl+h]
        ax.add_patch(MplPoly(pts, facecolor=plane_color, edgecolor='none',
                             alpha=plane_alpha, zorder=3))
        ax.add_patch(MplPoly(pts, facecolor='none', edgecolor=plane_color,
                             linewidth=1.0, alpha=0.55, zorder=3.1))

    # pass 3: redraw front so it occludes the plane
    for z2v, z1, z3, pdf, sx_f, sy_f, sx, sy in strips:
        if z2v >= z2c:
            continue
        for k in range(n_sub):
            avg = 0.5*(pdf[k]+pdf[k+1])
            c   = _soften(cmap_fn(avg), (1,.35,.35), 0.10)
            ax.add_patch(MplPoly(list(zip(
                [sx[k],sx[k+1],sx_f[k+1],sx_f[k]],
                [sy[k],sy[k+1],sy_f[k+1],sy_f[k]])),
                facecolor=c, edgecolor='none', linewidth=0, alpha=0.14, zorder=4))

    ax.set_xlim(-2.55, 4.05)
    ax.set_ylim(-1.85, 3.65)


# ── preview ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4.5, 4.2))
draw_dirichlet(ax, n_strips=80, n_sub=60)   # fast preview; full quality in assembly
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 6 — Element: stimulus thumbnail + shared top row
# ─────────────────────────────────────────────────────────────────
import os
from pathlib import Path

# ── knob: path to a stimulus image (swap freely) ──────────────────
STIM_IMG_PATH = Path('../stimuli/imgs/prototype_imgs') / \
    '000457_shico_f04_yaw016_s043_lab069-+040--038_t008_gl081_xy+0149-+0017_random.png'

STIM_LABEL = r'Stimulus $x_7$'    # label shown above the thumbnail
STIM_LABEL_COLOR = '#00b0b9'       # teal


def draw_stimulus(ax, img_path=None, label=None, label_color=None,
                  border_color='black', border_lw=1.5):
    """
    Show a stimulus thumbnail in ax.
    If img_path is missing, a grey placeholder box is drawn.
    """
    img_path    = Path(img_path or STIM_IMG_PATH)
    label       = label       or STIM_LABEL
    label_color = label_color or STIM_LABEL_COLOR

    ax.axis('off')
    if img_path.exists():
        img = plt.imread(str(img_path))
        ax.imshow(img)
    else:
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.1, 0.1), 0.8, 0.8,
            boxstyle='round,pad=0.02',
            facecolor='#dddddd', edgecolor=border_color, lw=border_lw,
            transform=ax.transAxes))
        ax.text(0.5, 0.5, '?', transform=ax.transAxes,
                ha='center', va='center', fontsize=18, color='0.5')

    # label above
    ax.set_title(label, color=label_color,
                 fontsize=CFG['fs_desc'] + 1,
                 fontweight='bold', pad=3)


# ── preview ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(1.5, 1.8))
draw_stimulus(ax)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 7 — Element: P̂(yes | t, x) label row
# ─────────────────────────────────────────────────────────────────

def draw_phat_row(ax, n_tasks=3, colors=None, dots=True, fs=None):
    """
    Draw a row of P̂(yes|t,x) labels into a thin strip ax.
    n_tasks : number of task labels shown (last one always 'Task T')
    colors  : per-task colours; falls back to CFG['task_colors']
    dots    : whether to draw '...' between last named task and Task T
    """
    colors = colors or CFG['task_colors']
    fs     = fs     or CFG['fs_phat']
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    xs = np.linspace(0.12, 0.88, n_tasks)
    label = r'$\hat{P}(\mathrm{yes}\mid t, x)$'

    for i, (x, c) in enumerate(zip(xs, colors[:n_tasks])):
        ax.text(x, 0.5, label, ha='center', va='center',
                fontsize=fs, color='black',
                transform=ax.transAxes)

    if dots and n_tasks >= 3:
        # insert dots between second-to-last and last
        mid = (xs[-2] + xs[-1]) / 2
        ax.text(mid, 0.5, r'$\cdots$', ha='center', va='center',
                fontsize=fs + 3)


# ── preview ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 0.6))
draw_phat_row(ax)
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Cell 8 — Full-figure assembly
#
# Layout (figure-fraction coordinates):
#
#   ┌──────────────────────────────────────────┐
#   │         stimulus + Clip-frozen box       │  top_strip
#   ├─────────────────────┬────────────────────┤
#   │  SLDA (purple)      │  DLBT (red)        │
#   │  Attpool box        │  Attpool box       │
#   │  CLIP cloud         │  CLIP cloud        │
#   │  "one decoder/task" │  Mapper box        │
#   │  3× task panels     │  Dirichlet surface │
#   │  3× sigmoids        │                   │
#   ├─────────────────────┴────────────────────┤
#   │      P̂(yes|t,x)  ···  P̂(yes|t,x)       │  bot_strip
#   └──────────────────────────────────────────┘
# ─────────────────────────────────────────────────────────────────

# ── layout knobs ─────────────────────────────────────────────────
FW, FH   = CFG['fig_w'], CFG['fig_h']
GAP      = CFG['col_gap']       # gap between SLDA and DLBT boxes
PAD      = CFG['box_pad']       # inner margin inside each box
TOP      = CFG['top_strip']     # shared header height (fraction)
BOT      = CFG['bot_strip']     # P̂ row height (fraction)

# vertical sub-divisions inside each column (fractions of the inner height)
INNER_H  = 1.0 - TOP - BOT
ROW_ATTPOOL  = 0.09    # Attpool box
ROW_CLOUD    = 0.32    # CLIP cloud
ROW_MIDDLE   = 0.09    # "one decoder / Mapper" label row
ROW_CONTENT  = 0.35    # task panels / Dirichlet
ROW_SIG      = 0.15    # sigmoid row (SLDA only; not used in DLBT)
# SLDA: attpool + cloud + middle + task + sig == 1.0
# DLBT: attpool + cloud + middle + dirichlet  == 1.0

fig = plt.figure(figsize=(FW, FH))

# ── helper: add_ax(left, bottom, width, height) in figure coords ──
def A(l, b, w, h): return fig.add_axes([l, b, w, h])

# Column boundaries
left_L  = 0.01
right_L = 0.5 - GAP/2
col_w   = right_L - left_L

left_R  = 0.5 + GAP/2
right_R = 0.99
col_wR  = right_R - left_R

# ── 0. outer coloured boxes (drawn on a background axes) ──────────
ax_bg = fig.add_axes([0, 0, 1, 1])
ax_bg.set_xlim(0, 1); ax_bg.set_ylim(0, 1)
ax_bg.axis('off')
ax_bg.set_zorder(-10)
fig.patch.set_facecolor('white')

box_bottom = BOT
box_height = INNER_H

for bx, bw, bc in [(left_L, col_w, SLDA_C), (left_R, col_wR, DLBT_C)]:
    ax_bg.add_patch(FancyBboxPatch(
        (bx, box_bottom), bw, box_height,
        boxstyle=f'round,pad=0,rounding_size={CFG["box_rounding"]}',
        facecolor='none', edgecolor=bc,
        linewidth=CFG['box_lw'],
        transform=ax_bg.transAxes, clip_on=False, zorder=1))

# ── 1. Shared top row ─────────────────────────────────────────────
# 1a. Column titles
for x, label, color in [
    (left_L + col_w*0.5,  'Standard Linear Decoder Approach (SLDA)', SLDA_C),
    (left_R + col_wR*0.5, 'Deep Latent Belief Tomography (DLBT)',    DLBT_C),
]:
    ax_bg.text(x, box_bottom + box_height + 0.01, label,
               ha='center', va='bottom', color=color,
               fontsize=CFG['fs_title'], fontweight='bold',
               transform=ax_bg.transAxes)

# 1b. Stimulus thumbnail (centred)
stim_w = 0.08; stim_h = TOP * 0.65
ax_stim = A(0.5 - stim_w/2, 1 - TOP + TOP*0.22, stim_w, stim_h)
draw_stimulus(ax_stim)

# 1c. "Clip frozen" label box  (centred, below stimulus)
ax_clip = A(0.38, 1 - TOP + 0.005, 0.24, TOP * 0.18)
ax_clip.axis('off')
black_label_box(ax_clip, (0.5, 0.5), 'Clip frozen', fs=CFG['fs_box'])

# 1d. Down-arrow from stimulus to Clip box, then two arrows to the columns
# (drawn on ax_bg using figure-fraction coords)
stim_cx = 0.5
clip_top = 1 - TOP + 0.005 + TOP*0.18
clip_bot = 1 - TOP + 0.005

arrow(ax_bg, (stim_cx, 1-TOP+0.22), (stim_cx, clip_top + 0.002),
      lw=1.5, ms=10, zorder=5)
# two arrows from clip box to the two column Attpool boxes
attpool_y = BOT + INNER_H - INNER_H*ROW_ATTPOOL*0.5
for col_cx in [left_L + col_w*0.5, left_R + col_wR*0.5]:
    ax_bg.annotate('',
        xy=(col_cx, BOT + INNER_H*(1-ROW_ATTPOOL) + INNER_H*ROW_ATTPOOL*0.5),
        xytext=(stim_cx, clip_bot),
        arrowprops=dict(arrowstyle='->', lw=1.5, color='black',
                        connectionstyle='arc3,rad=0.0',
                        shrinkA=2, shrinkB=2, mutation_scale=12),
        xycoords='axes fraction', textcoords='axes fraction', zorder=5)


# ── helper: vertical position within a column ─────────────────────
def col_row(col_left, col_w, row_bottom_frac, row_h_frac, inner_pad=0.008):
    """Return (left, bottom, width, height) in figure-fraction for a row."""
    l = col_left + inner_pad
    w = col_w    - 2*inner_pad
    b = BOT + INNER_H * row_bottom_frac + inner_pad
    h = INNER_H * row_h_frac  - 2*inner_pad
    return l, b, w, h


# ── 2. SLDA column ────────────────────────────────────────────────
# cumulative bottom fractions inside [0, INNER_H]
sig_bot   = 0.0
task_bot  = sig_bot  + ROW_SIG
mid_bot   = task_bot + ROW_CONTENT
cloud_bot = mid_bot  + ROW_MIDDLE
att_bot   = cloud_bot + ROW_CLOUD

# Attpool FT box
ax_att_L = A(*col_row(left_L, col_w, att_bot, ROW_ATTPOOL))
ax_att_L.axis('off')
black_label_box(ax_att_L, (0.5, 0.5), 'Attpool FT', fs=CFG['fs_box'])

# CLIP cloud
ax_cloud_L = A(*col_row(left_L, col_w, cloud_bot, ROW_CLOUD))
draw_latent_space(ax_cloud_L,
                  title=r'CLIP feature space $f(x) \in \mathbb{R}^{1024}$',
                  annotation=(0.72, 1.95))

# "One linear decoder per task" label
ax_mid_L = A(*col_row(left_L, col_w, mid_bot, ROW_MIDDLE))
ax_mid_L.axis('off')
ax_mid_L.text(0.5, 0.5,
              'One linear decoder per task\n(fit independently)',
              ha='center', va='center', fontsize=CFG['fs_desc'],
              fontweight='bold', transform=ax_mid_L.transAxes)

# Three task panels
n_tasks   = 3
task_xs   = np.linspace(left_L + 0.01, right_L - 0.01, n_tasks + 1)
task_pw   = (task_xs[1] - task_xs[0]) * 0.82

for i, (name, color, edge) in enumerate(
        zip(CFG['task_names'], CFG['task_colors'], CFG['task_edges'])):
    cx = task_xs[i] + (task_xs[i+1]-task_xs[i])/2 - task_pw/2
    # cloud + plane
    ax_tp = A(cx,
              BOT + INNER_H*(task_bot + ROW_SIG) + 0.005,
              task_pw,
              INNER_H*ROW_CONTENT * 0.52 - 0.01)
    draw_task_panel(ax_tp, name, color, edge,
                    rng=np.random.default_rng(CFG['cloud_seed']))
    # sigmoid below
    ax_sg = A(cx,
              BOT + INNER_H*sig_bot + 0.005,
              task_pw,
              INNER_H*ROW_SIG - 0.01)
    draw_sigmoid(ax_sg, color=color, k=CFG['task_k'][i])
    # equation label between panel and sigmoid
    ax_eq = A(cx, BOT + INNER_H*(task_bot + ROW_SIG*0.52), task_pw,
              INNER_H*ROW_CONTENT * 0.46)
    ax_eq.axis('off')
    ax_eq.text(0.5, 0.5,
               r'$\sigma(w^\top f(x) + b)$',
               ha='center', va='center', fontsize=CFG['fs_desc'] - 1)

# '...' dots between Task 2 panel and Task T
dots_cx = (task_xs[2] + task_xs[3])/2 - 0.01
ax_dots = A(dots_cx, BOT + INNER_H*(task_bot + ROW_SIG) + 0.01,
            0.02, INNER_H*ROW_CONTENT*0.52)
ax_dots.axis('off')
ax_dots.text(0.5, 0.5, r'$\cdots$', ha='center', va='center', fontsize=16)


# ── 3. DLBT column ────────────────────────────────────────────────
dir_bot   = 0.0
midR_bot  = dir_bot   + ROW_CONTENT + ROW_SIG   # reuse same height
cloudR_bot = midR_bot + ROW_MIDDLE
attR_bot  = cloudR_bot + ROW_CLOUD

# Attpool FT box
ax_att_R = A(*col_row(left_R, col_wR, attR_bot, ROW_ATTPOOL))
ax_att_R.axis('off')
black_label_box(ax_att_R, (0.5, 0.5), 'Attpool FT', fs=CFG['fs_box'])

# CLIP cloud
ax_cloud_R = A(*col_row(left_R, col_wR, cloudR_bot, ROW_CLOUD))
draw_latent_space(ax_cloud_R,
                  title=r'CLIP feature space $f(x) \in \mathbb{R}^{1024}$',
                  annotation=(0.72, 1.95))

# Mapper box
ax_mid_R = A(*col_row(left_R, col_wR, midR_bot, ROW_MIDDLE))
ax_mid_R.axis('off')
black_label_box(ax_mid_R, (0.5, 0.5), 'Mapper', fs=CFG['fs_box'])

# Dirichlet surface (takes the content + sigmoid height combined)
ax_dir = A(*col_row(left_R, col_wR, dir_bot, ROW_CONTENT + ROW_SIG))
draw_dirichlet(ax_dir)   # uses full quality from CFG


# ── 4. Bottom P̂ row (shared) ─────────────────────────────────────
ax_pL = A(left_L, 0.005, col_w,  BOT - 0.01)
ax_pR = A(left_R, 0.005, col_wR, BOT - 0.01)
draw_phat_row(ax_pL, n_tasks=3, colors=CFG['task_colors'])
draw_phat_row(ax_pR, n_tasks=3, colors=[DLBT_C]*3)


# ── 5. Save ───────────────────────────────────────────────────────
OUT_SVG = 'figure3_schematic.svg'
OUT_PDF = 'figure3_schematic.pdf'

fig.savefig(OUT_SVG, format='svg', bbox_inches='tight', facecolor='white')
fig.savefig(OUT_PDF, format='pdf', bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved → {OUT_SVG}  and  {OUT_PDF}')